# LoRA-style Matrix Factorization – REES46

## Ý tưởng
LoRA trong LLM: thay vì update W (lớn), học **ΔW = A · B** với rank thấp.  
Ở đây mình áp dụng y chang cho ma trận tương tác:

```
R  ≈  A  ·  B
(U×I)  (U×r) (r×I)
```

- `A` = user embedding matrix  (267k × r)
- `B` = item embedding matrix  (r × 60k)
- `r` = rank rất nhỏ: 8, 16, 32 → **giảm chiều mạnh**

**Khởi tạo kiểu LoRA:**
- `A` ~ Gaussian (random)
- `B` = 0 → lúc đầu dự đoán = 0, học dần lên

**Loss:** Mean Squared Error chỉ trên các cặp đã tương tác (observed entries)

**Metric:** Recall@20, Recall@50

In [1]:
!git clone https://huggingface.co/datasets/nguyenmaiductrong/rees46-processed-data

Cloning into 'rees46-processed-data'...
remote: Enumerating objects: 25, done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 25 (from 1)
Receiving objects: 100% (25/25), 1.77 MiB | 6.23 MiB/s, done.
Filtering content: 100% (13/13), 675.87 MiB | 291.41 MiB/s, done.


In [2]:
import numpy as np
import json, os, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import coo_matrix, csr_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

data_dir = "/content/rees46-processed-data/temporal"

Device: cuda


## 1. Load dữ liệu

In [3]:
view_src     = np.load(f"{data_dir}/view_train_src.npy")
view_dst     = np.load(f"{data_dir}/view_train_dst.npy")
cart_src     = np.load(f"{data_dir}/cart_train_src.npy")
cart_dst     = np.load(f"{data_dir}/cart_train_dst.npy")
purchase_src = np.load(f"{data_dir}/purchase_train_src.npy")
purchase_dst = np.load(f"{data_dir}/purchase_train_dst.npy")

val_user     = np.load(f"{data_dir}/val_user_idx.npy")
val_product  = np.load(f"{data_dir}/val_product_idx.npy")
test_user    = np.load(f"{data_dir}/test_user_idx.npy")
test_product = np.load(f"{data_dir}/test_product_idx.npy")

with open(f"{data_dir}/node_counts.json") as f:
    node_counts = json.load(f)

NUM_USERS = node_counts['user']    # 267,973
NUM_ITEMS = node_counts['product'] # 60,798
print(f"Users: {NUM_USERS:,} | Items: {NUM_ITEMS:,}")

Users: 267,973 | Items: 60,798


## 2. Chuẩn bị training data

Chỉ train trên các cặp **đã tương tác** (sparse observed entries)  
→ không cần load cả ma trận R lên RAM

In [4]:
# Gộp tất cả interactions + trọng số
all_users = np.concatenate([view_src, cart_src, purchase_src]).astype(np.int32)
all_items = np.concatenate([view_dst, cart_dst, purchase_dst]).astype(np.int32)
all_weights = np.concatenate([
    np.ones(len(view_src),     dtype=np.float32),
    np.ones(len(cart_src),     dtype=np.float32) * 2,
    np.ones(len(purchase_src), dtype=np.float32) * 3,
])

# Normalize weights về [0, 1] để loss ổn định hơn
all_weights = all_weights / all_weights.max()

print(f"Total interactions: {len(all_users):,}")
print(f"Weight range: [{all_weights.min():.2f}, {all_weights.max():.2f}]")

Total interactions: 42,884,336
Weight range: [0.33, 1.00]


In [5]:
class InteractionDataset(Dataset):
    def __init__(self, users, items, weights):
        self.users   = torch.LongTensor(users)
        self.items   = torch.LongTensor(items)
        self.weights = torch.FloatTensor(weights)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.weights[idx]


dataset = InteractionDataset(all_users, all_items, all_weights)
loader  = DataLoader(dataset, batch_size=65536, shuffle=True, num_workers=2, pin_memory=True)
print(f"Batches per epoch: {len(loader)}")

Batches per epoch: 655


## 3. Model – LoRA-style Matrix Factorization

```
        LoRA                     Standard MF
  A (U×r)  B (r×I)          E_user (U×d)  E_item (I×d)
  init: A~N(0,σ), B=0        init: ~N(0,σ)
  params: U·r + r·I          params: U·d + I·d
```

Với r=16, d=64:
- LoRA:  267k×16 + 16×60k  = **5.2M params**  
- Thông thường: 267k×64 + 60k×64 = **21M params**  ✓ nhỏ hơn 4×

In [6]:
class LoRAMatrixFactorization(nn.Module):
    """
    Score(u, i) = <A[u], B[i]>  với  A=user_lora, B=item_lora

    Khởi tạo đúng kiểu LoRA:
      - A ~ N(0, 1/sqrt(rank))  (Kaiming-style)
      - B = 0  →  output ban đầu = 0, học dần
    """
    def __init__(self, num_users, num_items, rank=16):
        super().__init__()
        self.rank = rank

        # A matrix: user side
        self.A = nn.Embedding(num_users, rank)
        nn.init.normal_(self.A.weight, std=1.0 / np.sqrt(rank))

        # B matrix: item side – khởi tạo = 0 (LoRA style)
        self.B = nn.Embedding(num_items, rank)
        nn.init.zeros_(self.B.weight)

        # Bias (optional nhưng giúp học nhanh hơn)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids):
        a = self.A(user_ids)           # (batch, rank)
        b = self.B(item_ids)           # (batch, rank)
        dot = (a * b).sum(dim=-1)      # (batch,)  – dot product

        ub = self.user_bias(user_ids).squeeze(-1)
        ib = self.item_bias(item_ids).squeeze(-1)

        return dot + ub + ib

    def get_all_embeddings(self):
        """Trả về toàn bộ embedding để tính Recall."""
        user_emb = self.A.weight.detach().cpu().numpy()  # (U, rank)
        item_emb = self.B.weight.detach().cpu().numpy()  # (I, rank)
        return user_emb, item_emb


RANK = 16  # ← thử 8, 16, 32
model = LoRAMatrixFactorization(NUM_USERS, NUM_ITEMS, rank=RANK).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Rank: {RANK}")
print(f"Total params: {total_params:,} ({total_params/1e6:.1f}M)")

Rank: 16
Total params: 5,589,107 (5.6M)


## 4. Training

In [7]:
def recall_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=512):
    hits = 0
    n = len(user_idx)
    item_emb_T = item_emb.T
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores = user_emb[u_batch] @ item_emb_T
        top_k  = np.argpartition(-scores, k, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            if gt in top_k[i]:
                hits += 1
    return hits / n


def ndcg_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=512):
    """
    NDCG@K – mỗi user chỉ có 1 ground-truth item.
    """
    total = 0.0
    n = len(user_idx)
    item_emb_T = item_emb.T
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores   = user_emb[u_batch] @ item_emb_T
        top_k_idx = np.argsort(-scores, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            ranks = np.where(top_k_idx[i] == gt)[0]
            if len(ranks) > 0:
                total += 1.0 / np.log2(ranks[0] + 2)
    return total / n


def evaluate_all(user_emb, item_emb, user_idx, true_items, label="", batch_size=512):
    """In đầy đủ Recall@10/20/50 và NDCG@10/20/50."""
    r10 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    r20 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    r50 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    n10 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    n20 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    n50 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    hdr = f"  {label}  " if label else "  "
    print(f"\n{'='*52}")
    print(f"{hdr}Recall@10  : {r10:.4f}")
    print(f"{hdr}Recall@20  : {r20:.4f}")
    print(f"{hdr}Recall@50  : {r50:.4f}")
    print(f"{hdr}NDCG@10    : {n10:.4f}")
    print(f"{hdr}NDCG@20    : {n20:.4f}")
    print(f"{hdr}NDCG@50    : {n50:.4f}")
    print(f"{'='*52}")
    return dict(r10=r10, r20=r20, r50=r50, n10=n10, n20=n20, n50=n50)


In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS   = 10
best_r20 = 0.0
best_emb = None

print(f"{'Epoch':>5} | {'Loss':>10} | {'R@10':>8} | {'R@20':>8} | {'R@50':>8} | {'N@10':>8} | {'N@20':>8} | {'N@50':>8} | {'Time':>7}")
print("-" * 90)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for users, items, targets in loader:
        users   = users.to(device)
        items   = items.to(device)
        targets = targets.to(device)
        preds = model(users, items)
        loss = (targets * (preds - targets) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    model.eval()
    with torch.no_grad():
        ue, ie = model.get_all_embeddings()

    r10 = recall_at_k(ue, ie, val_user, val_product, k=10)
    r20 = recall_at_k(ue, ie, val_user, val_product, k=20)
    r50 = recall_at_k(ue, ie, val_user, val_product, k=50)
    n10 = ndcg_at_k  (ue, ie, val_user, val_product, k=10)
    n20 = ndcg_at_k  (ue, ie, val_user, val_product, k=20)
    n50 = ndcg_at_k  (ue, ie, val_user, val_product, k=50)
    elapsed = time.time() - t0

    print(f"{epoch:>5} | {total_loss/len(loader):>10.4f} | {r10:>8.4f} | {r20:>8.4f} | {r50:>8.4f} | {n10:>8.4f} | {n20:>8.4f} | {n50:>8.4f} | {elapsed:>5.1f}s")

    if r20 > best_r20:
        best_r20 = r20
        best_emb = (ue.copy(), ie.copy())

print(f"\nBest Val Recall@20: {best_r20:.4f}")


Epoch |       Loss |     R@10 |     R@20 |     R@50 |     N@10 |     N@20 |     N@50 |    Time
------------------------------------------------------------------------------------------
    1 |     0.0537 |   0.0474 |   0.0698 |   0.1104 |   0.0245 |   0.0301 |   0.0381 | 1364.0s
    2 |     0.0294 |   0.0938 |   0.1248 |   0.1761 |   0.0547 |   0.0626 |   0.0727 | 1361.9s
    3 |     0.0289 |   0.1050 |   0.1352 |   0.1782 |   0.0616 |   0.0692 |   0.0778 | 1368.3s
    4 |     0.0288 |   0.1018 |   0.1315 |   0.1695 |   0.0607 |   0.0682 |   0.0757 | 1367.9s
    5 |     0.0288 |   0.0947 |   0.1231 |   0.1571 |   0.0558 |   0.0629 |   0.0697 | 1358.7s
    6 |     0.0288 |   0.0862 |   0.1147 |   0.1489 |   0.0495 |   0.0566 |   0.0634 | 1358.3s
    7 |     0.0288 |   0.0886 |   0.1194 |   0.1540 |   0.0506 |   0.0583 |   0.0653 | 1363.5s
    8 |     0.0288 |   0.0841 |   0.1137 |   0.1484 |   0.0467 |   0.0541 |   0.0611 | 1366.9s
    9 |     0.0288 |   0.0814 |   0.1106 |   0.1465 | 

## 5. Kết quả cuối trên Test set

In [9]:
ue_best, ie_best = best_emb

print("Kết quả cuối trên Test set (best val checkpoint):")
test_metrics = evaluate_all(ue_best, ie_best, test_user, test_product, label="TEST")
print(f"  Rank used       : {RANK}")


Kết quả cuối trên Test set (best val checkpoint):

  TEST  Recall@10  : 0.0485
  TEST  Recall@20  : 0.0702
  TEST  Recall@50  : 0.1018
  TEST  NDCG@10    : 0.0253
  TEST  NDCG@20    : 0.0308
  TEST  NDCG@50    : 0.0371
  Rank used       : 16


## 6. So sánh các Rank (optional)

Thử rank = 8, 16, 32 để tìm điểm tối ưu

In [10]:
print(f"{'Rank':>6} | {'Params':>9} | {'R@10':>8} | {'R@20':>8} | {'R@50':>8} | {'N@10':>8} | {'N@20':>8} | {'N@50':>8}")
print("-" * 80)

for rank in [8, 16, 32]:
    m = LoRAMatrixFactorization(NUM_USERS, NUM_ITEMS, rank=rank).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5)

    for _ in range(5):
        m.train()
        for users, items, targets in loader:
            users, items, targets = users.to(device), items.to(device), targets.to(device)
            loss = (targets * (m(users, items) - targets) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()

    m.eval()
    with torch.no_grad():
        ue_, ie_ = m.get_all_embeddings()

    r10 = recall_at_k(ue_, ie_, val_user, val_product, k=10)
    r20 = recall_at_k(ue_, ie_, val_user, val_product, k=20)
    r50 = recall_at_k(ue_, ie_, val_user, val_product, k=50)
    n10 = ndcg_at_k  (ue_, ie_, val_user, val_product, k=10)
    n20 = ndcg_at_k  (ue_, ie_, val_user, val_product, k=20)
    n50 = ndcg_at_k  (ue_, ie_, val_user, val_product, k=50)
    params = sum(p.numel() for p in m.parameters())

    print(f"{rank:>6} | {params:>9,} | {r10:>8.4f} | {r20:>8.4f} | {r50:>8.4f} | {n10:>8.4f} | {n20:>8.4f} | {n50:>8.4f}")


  Rank |    Params |     R@10 |     R@20 |     R@50 |     N@10 |     N@20 |     N@50
--------------------------------------------------------------------------------
     8 | 2,958,939 |   0.0942 |   0.1249 |   0.1589 |   0.0535 |   0.0612 |   0.0680
    16 | 5,589,107 |   0.0794 |   0.1085 |   0.1421 |   0.0427 |   0.0501 |   0.0569
    32 | 10,849,443 |   0.0795 |   0.1107 |   0.1440 |   0.0413 |   0.0491 |   0.0558
